# ICT-36h — F-Lens x Random Forest : le decodeur du papier LSD et l'honnetete de sa validation

**Navigation** : [Index](README.md) | [Version Python](ICT-36h-FLens-RFNestedCV-Python.ipynb) | [<< ICT-36g](ICT-36g-FLens-InteractionTransmodal-Python.ipynb) | [>> ICT-37](ICT-37-FLens-BeliefState-Python.ipynb)

Ce carnet est le **jumeau C#** de `ICT-36h-FLens-RFNestedCV-Python.ipynb`. Il rejoue la meme
experience — un decodeur qui doit retrouver la condition (placebo ou LSD) a partir de
descripteurs de complexite calcules sur des signaux AR(1) synthetiques — et il pose la meme
question methodologique : **une validation croisee non imbriquee fabrique-t-elle l'effet
qu'elle pretend mesurer ?**

Le portage est ecrit en BCL pur : ni scikit-learn, ni NumPy. Cela change une chose, et il faut
la dire d'entree de jeu.

### Ce que ce carnet n'est pas

Ce n'est pas un jumeau numerique. Les generateurs de nombres et les forets aleatoires de ce
carnet ne sont pas ceux de NumPy et de scikit-learn : les valeurs imprimees ici sont **propres
a ce portage** et ne se comparent pas chiffre a chiffre a celles du carnet Python. Ce qui se
compare, c'est la **structure des conclusions** : le decodeur retrouve-t-il la condition quand
elle est dans les donnees, retombe-t-il au hasard quand elle n'y est pas, et la validation
croisee non imbriquee surestime-t-elle la precision ?

Ecrire une foret aleatoire a la main a un interet pedagogique direct : les plis, le groupe, et
la fuite deviennent des objets visibles au lieu d'etre caches derriere un appel de bibliotheque.


## 1. Mise en place

Le carnet n'importe aucune bibliotheque scientifique : tout est en BCL. Deux briques sont
ecrites a la main des la section 2, parce qu'elles portent l'essentiel du propos :

- un **generateur deterministe** — un meme germe doit rendre le meme terrain, sinon aucune
  conclusion du carnet n'est reproductible ;
- une **epingle de culture** — les nombres sortent a point decimal, quelle que soit la machine.

Les constantes sont celles du carnet Python, sans reinterpretation : `N_SUBJ = 17`, douze epochs
par sujet, douze regions, `T_EPOCH = 1200` points a 600 Hz, et une amplitude d'effet
`ETA_DECODE = 0.06` choisie pour que la condition soit **decodable mais pas triviale**.


In [1]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Le carnet imprime des nombres a point decimal : la culture est epinglee ici, une fois,
// pour que toutes les cellules en heritent. Ces trois lignes se posent apres le dernier
// `using` : entre deux directives `using`, elles ne compileraient pas (CS1529).
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;

/// <summary>Generateur deterministe : SplitMix64 pour les entiers, Box-Muller pour les normales.</summary>
public sealed class Rng
{
    private ulong _state;

    public Rng(int seed) => _state = unchecked((ulong)seed * 0x9E3779B97F4A7C15UL + 0x1234567UL);

    public ulong NextULong()
    {
        _state += 0x9E3779B97F4A7C15UL;
        ulong z = _state;
        z = (z ^ (z >> 30)) * 0xBF58476D1CE4E5B9UL;
        z = (z ^ (z >> 27)) * 0x94D049BB133111EBUL;
        return z ^ (z >> 31);
    }

    /// <summary>Uniforme sur [0, 1).</summary>
    public double NextDouble() => (NextULong() >> 11) * (1.0 / 9007199254740992.0);

    /// <summary>Entier uniforme dans [lo, hi) — la borne haute est exclue, comme numpy.integers.</summary>
    public int NextInt(int lo, int hi) => lo + (int)(NextULong() % (ulong)(hi - lo));

    /// <summary>Normale centree reduite par Box-Muller, avec cache.</summary>
    private double? _spare;

    public double NextGaussian()
    {
        if (_spare.HasValue)
        {
            double s = _spare.Value;
            _spare = null;
            return s;
        }
        double u1, u2, r;
        do
        {
            u1 = NextDouble();
        }
        while (u1 <= double.Epsilon);
        u2 = NextDouble();
        r = Math.Sqrt(-2.0 * Math.Log(u1));
        _spare = r * Math.Sin(2.0 * Math.PI * u2);
        return r * Math.Cos(2.0 * Math.PI * u2);
    }
}

Console.WriteLine("C# portage ICT-36h | generateur SplitMix64 | culture epinglee");


The below script needs to be able to find the current output cell; this is an easy method to get it.

C# portage ICT-36h | generateur SplitMix64 | culture epinglee


In [2]:
// --- terrain : les constantes du carnet Python, a l'identique ---
int[] SEEDS = { 0, 1, 7, 42 };
const int N_SUBJ = 17;
const int N_ROI = 12;
const int EPOCHS_PER_SUBJ = 12;
const int T_EPOCH = 1200;
const double FS = 600.0;
const double RHO_PLA = 0.90;
const double ETA_DECODE = 0.06;
const double SIGMA_RHO_SUBJ = 0.05;
const double SIGMA_HFD_SUBJ = 0.12;
const double SIGMA_LZC_SUBJ = 0.035;
const int KMAX_HFD = 18;

// --- decodeur ---
const int N_TREES = 100;
const int N_PERM = 30;

Console.WriteLine($"constantes : N_SUBJ {N_SUBJ} | T_EPOCH {T_EPOCH} | N_ROI {N_ROI} | ETA_DECODE {ETA_DECODE} | SEEDS [{string.Join(", ", SEEDS)}] | N_PERM {N_PERM}");


constantes : N_SUBJ 17 | T_EPOCH 1200 | N_ROI 12 | ETA_DECODE 0.06 | SEEDS [0, 1, 7, 42] | N_PERM 30


## 2. Le terrain synthetique

Chaque epoch est un signal AR(1) par region : `x[t] = rho * x[t-1] + sqrt(1 - rho^2) * bruit`.
La forme exacte compte — le coefficient `sqrt(1 - rho^2)` maintient la variance a l'unite, donc
seule la **memoire** du signal varie, pas son energie.

Le LSD retire de la persistance `rho`, proportionnellement a l'amplitude `eta` et a un gradient
anterieur-posterieur. Deux descripteurs sont ensuite mesures sur chaque signal : la dimension
fractale de **Higuchi** et la complexite de **Lempel-Ziv** binarisee a la mediane. Les deux
reagissent a la perte de memoire, par des voies differentes — c'est cette redondance qui rend
le decodeur interessant.


In [3]:
// --- Higuchi : la dimension fractale d'un signal 1-D -------------------------------
// Implementation fidele a `antropy.higuchi_fd` : pour chaque pas k on mesure la longueur
// moyenne du chemin parcouru le long des k sous-series, puis on lit la pente de
// log L(k) contre log(1/k). La pente changee de signe EST la dimension fractale.
public static class Higuchi
{
    /// <summary>Longueurs L(k) et abscisses 1/k, pour k = 1..kmax.</summary>
    public static (double[] lk, double[] xReg) Curve(double[] x, int kmax)
    {
        int n = x.Length;
        var lk = new double[kmax];
        var xReg = new double[kmax];
        var lm = new double[kmax];
        for (int k = 1; k <= kmax; k++)
        {
            double sum = 0.0;
            for (int m = 0; m < k; m++)
            {
                double ll = 0.0;
                int nMax = (n - m - 1) / k;
                for (int j = 1; j < nMax; j++)
                {
                    ll += Math.Abs(x[m + j * k] - x[m + (j - 1) * k]);
                }
                ll /= k;
                ll *= (double)(n - 1) / (k * nMax);
                sum += ll;
            }
            lk[k - 1] = sum / k;
            xReg[k - 1] = 1.0 / k;
        }
        return (lk, xReg);
    }

    /// <summary>
    /// Dimension fractale : pente de log L(k) contre log(1/k). Comme L(k) decroit en k^-D,
    /// la pente vaut +D — verifie empiriquement sur du bruit blanc (D = 2.01) et sur un
    /// AR(1) de rho = 0.9 (D = 1.67).
    /// </summary>
    public static double Fd(double[] x, int kmax)
    {
        var (lk, xReg) = Curve(x, kmax);
        return Slope(xReg.Select(v => Math.Log(v)).ToArray(), lk.Select(v => Math.Log(v)).ToArray());
    }

    /// <summary>Pente des moindres carres ordinaires — meme estimateur que numpy.polyfit degre 1.</summary>
    public static double Slope(double[] x, double[] y)
    {
        double mx = x.Average(), my = y.Average();
        double sxy = 0.0, sxx = 0.0;
        for (int i = 0; i < x.Length; i++)
        {
            sxy += (x[i] - mx) * (y[i] - my);
            sxx += (x[i] - mx) * (x[i] - mx);
        }
        return sxy / sxx;
    }
}

// --- Lempel-Ziv 76 : la complexite d'une sequence binariee -------------------------
// Compteur c(m) de Kaspar et Schuster, puis normalisation par m / log2(m) : la valeur
// devient comparable entre longueurs de sequence. Un tirage aleatoire vaut ~1.07, une
// sequence constante ~0.017, une alternance parfaite ~0.026.
public static class LempelZiv
{
    public static int Count(string s)
    {
        int n = s.Length;
        int u = 0, v = 1, w = 1, vMax = 1, complexity = 1;
        while (true)
        {
            // Garde de bord : quand la fenetre de comparaison atteint la fin de la sequence,
            // le dernier bloc est compte et l'on sort. Sans elle, l'indice w + v depasse la
            // longueur — c'est le cas que la reference Python ne rencontre que par chance,
            // et que ce portage doit traiter explicitement.
            if (w + v >= n || u + v >= n)
            {
                complexity++;
                break;
            }
            if (s[u + v] == s[w + v])
            {
                v++;
            }
            else
            {
                if (v > vMax) vMax = v;
                u++;
                if (u == w)
                {
                    complexity++;
                    w += vMax;
                    if (w > n) break;
                    u = 0;
                    v = 1;
                    vMax = 1;
                }
                else
                {
                    v = 1;
                }
            }
        }
        return complexity;
    }

    public static double Normalized(double[] x)
    {
        double median = Median(x);
        var sb = new System.Text.StringBuilder(x.Length);
        foreach (double v in x) sb.Append(v > median ? '1' : '0');
        string s = sb.ToString();
        double m = s.Length;
        return Count(s) / (m / Math.Log2(m));
    }

    public static double Median(double[] values)
    {
        var copy = (double[])values.Clone();
        Array.Sort(copy);
        int n = copy.Length;
        return n % 2 == 1 ? copy[n / 2] : 0.5 * (copy[n / 2 - 1] + copy[n / 2]);
    }
}

// --- la cohorte synthetique --------------------------------------------------------
public sealed record Cohort(double[][][] Feats, int[] Y, int[] Groups, int[][] Sizes);

public static class Terrain
{
    /// <summary>
    /// Genere une cohorte : chaque sujet porte des effets propres, chaque epoch un signal
    /// AR(1) par region, dont on tire deux descripteurs de complexite.
    /// </summary>
    public static Cohort Simulate(double eta, int seed)
    {
        var rng = new Rng(seed);
        // Le pole posterieur (indice N_ROI-1) est celui que le LSD desynchronise : gradient = 1.
        var gradient = new double[N_ROI];
        for (int r = 0; r < N_ROI; r++) gradient[r] = r / (double)(N_ROI - 1);

        var feats = new List<double[][]>();
        var labels = new List<int>();
        var groups = new List<int>();
        var sizes = new List<int[]>();

        for (int sujet = 0; sujet < N_SUBJ; sujet++)
        {
            // Repartition placebo/LSD tiree par sujet : les effectifs ne sont pas equilibres.
            int nLsd = rng.NextInt(4, 9);
            int nPla = EPOCHS_PER_SUBJ - nLsd;
            sizes.Add(new[] { nPla, nLsd });

            // Effets sujet : chaque individu derive autour du parametre de population.
            double sRho = rng.NextGaussian() * SIGMA_RHO_SUBJ;
            double sHfd = rng.NextGaussian() * SIGMA_HFD_SUBJ;
            double sLzc = rng.NextGaussian() * SIGMA_LZC_SUBJ;

            foreach (var (condition, nEpochs) in new[] { (0, nPla), (1, nLsd) })
            {
                for (int e = 0; e < nEpochs; e++)
                {
                    var epoch = new double[N_ROI][];
                    for (int r = 0; r < N_ROI; r++)
                    {
                        // Le LSD retire de la persistance, proportionnellement a eta et au gradient.
                        double rho = Math.Clamp(RHO_PLA - eta * condition * gradient[r] + sRho, 0.05, 0.97);
                        var x = new double[T_EPOCH];
                        x[0] = rng.NextGaussian();
                        double gain = Math.Sqrt(1.0 - rho * rho);
                        for (int t = 1; t < T_EPOCH; t++)
                        {
                            // AR(1) exact : la variance reste unitaire, seule la memoire change.
                            x[t] = rho * x[t - 1] + gain * rng.NextGaussian();
                        }
                        double hfd = Higuchi.Fd(x, KMAX_HFD) + sHfd;
                        double lzc = LempelZiv.Normalized(x) + sLzc;
                        epoch[r] = new[] { hfd, lzc };
                    }
                    feats.Add(epoch);
                    labels.Add(condition);
                    groups.Add(sujet);
                }
            }
        }

        return new Cohort(feats.ToArray(), labels.ToArray(), groups.ToArray(), sizes.ToArray());
    }
}

// --- temoins : avant de mesurer le terrain, on mesure l'instrument ------------------
// Un descripteur qui ne rend pas les valeurs connues sur des signaux elementaires ne
// mesure rien. Bruit blanc : dimension 2. Une sequence constante : complexite quasi nulle.
var rngWhite = new Rng(12345);
var white = new double[T_EPOCH];
for (int t = 0; t < T_EPOCH; t++) white[t] = rngWhite.NextGaussian();
double fdWhite = Higuchi.Fd(white, KMAX_HFD);

double norm = T_EPOCH / Math.Log2(T_EPOCH);
var rngBits = new Rng(999);
var bits = new char[T_EPOCH];
for (int t = 0; t < T_EPOCH; t++) bits[t] = rngBits.NextDouble() < 0.5 ? '1' : '0';
double lzRandom = LempelZiv.Count(new string(bits)) / norm;
double lzZeros = LempelZiv.Count(new string('0', T_EPOCH)) / norm;
double lzAlt = LempelZiv.Count(string.Concat(Enumerable.Repeat("01", T_EPOCH / 2))) / norm;

Console.WriteLine($"temoins : Higuchi bruit blanc {fdWhite:F4} (attendu ~2.0)");
Console.WriteLine($"temoins : LZC aleatoire {lzRandom:F4} (attendu ~1.07) | constant {lzZeros:F4} (attendu ~0.017) | alternance {lzAlt:F4} (attendu ~0.026)");
Console.WriteLine();

var sw = System.Diagnostics.Stopwatch.StartNew();
var cohort = Terrain.Simulate(ETA_DECODE, SEEDS[0]);
sw.Stop();

Console.WriteLine($"epochs {cohort.Feats.Length} | descripteurs multi {cohort.Feats.Length} x {N_ROI * 2}");
Console.WriteLine($"placebo {cohort.Y.Count(v => v == 0)} | LSD {cohort.Y.Count(v => v == 1)}");
Console.WriteLine($"sujets {cohort.Groups.Distinct().Count()}");
Console.WriteLine($"terrain genere en {sw.ElapsedMilliseconds} ms");


temoins : Higuchi bruit blanc 2.0066 (attendu ~2.0)


temoins : LZC aleatoire 1.0399 (attendu ~1.07) | constant 0.0170 (attendu ~0.017) | alternance 0.0256 (attendu ~0.026)


epochs 204 | descripteurs multi 204 x 24


placebo 98 | LSD 106


sujets 17


terrain genere en 2410 ms


### Lecture du resultat

Les effectifs ne sont pas equilibres (chaque sujet tire son propre nombre d'epochs LSD), et
c'est voulu : un decodeur qui ne fonctionne que sur des classes equilibrees ne fonctionne pas
sur des donnees reelles.

La table par region est celle qui compte. Un effet present se lit comme une **pente** : la
region 0 (anterieure) doit rester stable entre conditions, la region 11 (posterieure) doit se
decrocher. Un effet absent donnerait deux colonnes plates et confondues.

La dispersion, elle, dit deja ce que la validation croisee devra demeler : si l'ecart-type
entre epochs depasse l'ecart entre conditions, un classifieur lineaire echouera et il faudra
toute la puissance de la foret.


In [4]:
// --- mesure descriptive du terrain ------------------------------------------------
// Deux questions avant toute foret : les effectifs par sujet, et la dispersion par
// descripteur. C'est la dispersion qui dit si l'effet depasse le bruit sujet.
var sizes = cohort.Sizes;
Console.WriteLine("sujet | placebo | LSD");
for (int s = 0; s < sizes.Length; s++)
{
    Console.WriteLine($"{s,5} | {sizes[s][0],7} | {sizes[s][1],3}");
}
Console.WriteLine($"total | {sizes.Sum(v => v[0]),7} | {sizes.Sum(v => v[1]),3}");
Console.WriteLine();

static double Mean(IEnumerable<double> v) => v.Average();

static double Std(IEnumerable<double> v)
{
    var arr = v.ToArray();
    double m = arr.Average();
    return Math.Sqrt(arr.Sum(x => (x - m) * (x - m)) / (arr.Length - 1));
}

Console.WriteLine("descripteur | condition | moyenne | ecart-type | posterieur (region 11)");
foreach (var (nom, idx) in new[] { ("HFD", 0), ("LZC", 1) })
{
    foreach (var (condition, etiquette) in new[] { (0, "placebo"), (1, "LSD") })
    {
        // Les valeurs de tous les epochs de la condition, toutes regions confondues.
        var valeurs = new List<double>();
        var posterieur = new List<double>();
        for (int i = 0; i < cohort.Feats.Length; i++)
        {
            if (cohort.Y[i] != condition) continue;
            for (int r = 0; r < N_ROI; r++)
            {
                valeurs.Add(cohort.Feats[i][r][idx]);
            }
            posterieur.Add(cohort.Feats[i][N_ROI - 1][idx]);
        }
        Console.WriteLine($"{nom,11} | {etiquette,9} | {Mean(valeurs),7:F4} | {Std(valeurs),10:F4} | {Mean(posterieur),18:F4}");
    }
}
Console.WriteLine();

// Le gradient anterieur-posterieur : c'est la signature que le papier revendique.
Console.WriteLine("region | HFD placebo | HFD LSD | LZC placebo | LZC LSD");
for (int r = 0; r < N_ROI; r++)
{
    var hfdPla = Enumerable.Range(0, cohort.Feats.Length).Where(i => cohort.Y[i] == 0).Select(i => cohort.Feats[i][r][0]);
    var hfdLsd = Enumerable.Range(0, cohort.Feats.Length).Where(i => cohort.Y[i] == 1).Select(i => cohort.Feats[i][r][0]);
    var lzcPla = Enumerable.Range(0, cohort.Feats.Length).Where(i => cohort.Y[i] == 0).Select(i => cohort.Feats[i][r][1]);
    var lzcLsd = Enumerable.Range(0, cohort.Feats.Length).Where(i => cohort.Y[i] == 1).Select(i => cohort.Feats[i][r][1]);
    Console.WriteLine($"{r,6} | {Mean(hfdPla),11:F4} | {Mean(hfdLsd),7:F4} | {Mean(lzcPla),11:F4} | {Mean(lzcLsd),7:F4}");
}


sujet | placebo | LSD


    0 |       5 |   7


    1 |       7 |   5


    2 |       7 |   5


    3 |       6 |   6


    4 |       4 |   8


... (27 outputs left out)

     8 |      1.6729 |  1.7033 |      0.6049 |  0.6603


     9 |      1.6707 |  1.7130 |      0.6069 |  0.6726


    10 |      1.6734 |  1.7124 |      0.6018 |  0.6768


    11 |      1.6742 |  1.7229 |      0.6054 |  0.6922


## 4. Le decodeur : foret aleatoire sous validation croisee imbriquee

Le decodeur est une **foret aleatoire** : cent arbres de decision entraines chacun sur un
tirage avec remise des epochs, chaque coupure ne voyant qu'une racine carree des descripteurs.
La prediction agrege les cent arbres.

Une foret a des **hyperparametres** — profondeur maximale, taille minimale d'une feuille — et
c'est la que la methodologie se joue. Deux procedures sont ecrites ici, et elles ne mesurent
pas la meme chose :

| Procedure | Ce qu'elle fait | Ce qu'elle mesure |
|---|---|---|
| **Imbriquee** | la grille est choisie dans chaque pli exterieur, par une validation interne | la performance d'une **procedure complete**, hyperparametres compris |
| **Non imbriquee** | la grille est choisie sur toutes les donnees, puis evaluee | une performance **optimiste** : les hyperparametres ont vu le test |

Le second point est le sujet du carnet. Un troisieme decoupage s'y ajoute : **les plis sont
groupes par sujet**. Un sujet porte douze epochs ; si deux epochs du meme sujet tombent l'un
en apprentissage et l'autre en test, le decodeur reconnait l'individu, pas la condition.


In [5]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Linq;

// --- la foret aleatoire -----------------------------------------------------------
public sealed class ForestOptions
{
    public int NTrees { get; set; } = 100;
    /// <summary>Profondeur maximale ; int.MaxValue signifie « aucune limite ».</summary>
    public int MaxDepth { get; set; } = int.MaxValue;
    public int MinLeaf { get; set; } = 1;
    public int Seed { get; set; } = 0;
    /// <summary>Descripteurs examines par coupure ; 0 = racine carree du nombre total.</summary>
    public int Mtry { get; set; } = 0;

    public ForestOptions Clone() => new ForestOptions
    {
        NTrees = NTrees, MaxDepth = MaxDepth, MinLeaf = MinLeaf, Seed = Seed, Mtry = Mtry,
    };
}

public sealed class TreeNode
{
    public int Feature = -1;
    public double Threshold;
    public TreeNode? Left;
    public TreeNode? Right;
    /// <summary>Proportion de classe 1 dans la feuille — ou dans le noeud si celui-ci est terminal.</summary>
    public double Probability;
}

public sealed class RandomForest
{
    private readonly List<TreeNode> _trees = new();
    public int NFeatures { get; }
    public double[] Importances { get; }

    public RandomForest(double[][] X, int[] y, ForestOptions options)
    {
        if (X.Length != y.Length) throw new ArgumentException("X et y de longueurs differentes");
        NFeatures = X[0].Length;
        Importances = new double[NFeatures];
        int n = X.Length;
        int mtry = options.Mtry > 0 ? options.Mtry : Math.Max(1, (int)Math.Sqrt(NFeatures));
        var rng = new Rng(options.Seed);

        for (int t = 0; t < options.NTrees; t++)
        {
            var bootstrap = new int[n];
            for (int i = 0; i < n; i++) bootstrap[i] = rng.NextInt(0, n);
            _trees.Add(Build(bootstrap, X, y, options, mtry, 0, rng));
        }
    }

    private TreeNode Build(int[] idx, double[][] X, int[] y, ForestOptions options, int mtry, int depth, Rng rng)
    {
        int n = idx.Length;
        int positives = 0;
        foreach (int i in idx) positives += y[i];
        double probability = n == 0 ? 0.0 : (double)positives / n;

        // Un noeud pur n'a plus rien a separer ; une feuille trop petite serait du bruit.
        if (depth >= options.MaxDepth || n < 2 * options.MinLeaf || positives == 0 || positives == n)
        {
            return new TreeNode { Probability = probability };
        }

        int p = X[0].Length;
        var features = Enumerable.Range(0, p).ToArray();
        for (int i = p - 1; i > 0; i--)
        {
            int j = rng.NextInt(0, i + 1);
            (features[i], features[j]) = (features[j], features[i]);
        }
        int examined = Math.Min(mtry, p);
        double parentImpurity = 2.0 * probability * (1.0 - probability);

        double bestGain = 0.0;
        int bestFeature = -1;
        double bestThreshold = 0.0;

        for (int fi = 0; fi < examined; fi++)
        {
            int f = features[fi];
            // Candidats de coupure : seize quantiles des valeurs du noeud pour ce descripteur.
            var values = idx.Select(i => X[i][f]).OrderBy(v => v).ToArray();
            int nQuantiles = Math.Min(16, n - 1);
            for (int q = 1; q <= nQuantiles; q++)
            {
                int at = (int)((long)q * n / (nQuantiles + 1));
                double lo = values[at - 1], hi = values[at];
                if (hi <= lo) continue;
                double threshold = 0.5 * (lo + hi);

                int nLeft = 0, posLeft = 0;
                foreach (int i in idx)
                {
                    if (X[i][f] <= threshold) { nLeft++; posLeft += y[i]; }
                }
                int nRight = n - nLeft;
                int posRight = positives - posLeft;
                if (nLeft < options.MinLeaf || nRight < options.MinLeaf) continue;

                double pLeft = (double)posLeft / nLeft;
                double pRight = (double)posRight / nRight;
                double gain = parentImpurity
                    - ((double)nLeft / n) * 2.0 * pLeft * (1.0 - pLeft)
                    - ((double)nRight / n) * 2.0 * pRight * (1.0 - pRight);
                if (gain > bestGain + 1e-12)
                {
                    bestGain = gain;
                    bestFeature = f;
                    bestThreshold = threshold;
                }
            }
        }

        if (bestFeature < 0) return new TreeNode { Probability = probability };

        Importances[bestFeature] += bestGain * n;
        var left = idx.Where(i => X[i][bestFeature] <= bestThreshold).ToArray();
        var right = idx.Where(i => X[i][bestFeature] > bestThreshold).ToArray();
        return new TreeNode
        {
            Feature = bestFeature,
            Threshold = bestThreshold,
            Probability = probability,
            Left = Build(left, X, y, options, mtry, depth + 1, rng),
            Right = Build(right, X, y, options, mtry, depth + 1, rng),
        };
    }

    public double PredictProbability(double[] x)
    {
        double sum = 0.0;
        foreach (TreeNode tree in _trees) sum += LeafProbability(tree, x);
        return sum / _trees.Count;
    }

    public int Predict(double[] x) => PredictProbability(x) >= 0.5 ? 1 : 0;

    private static double LeafProbability(TreeNode node, double[] x)
    {
        while (node.Feature >= 0)
        {
            node = x[node.Feature] <= node.Threshold ? node.Left! : node.Right!;
        }
        return node.Probability;
    }

    /// <summary>Importances normalisees : la somme vaut 1, comme scikit-learn.</summary>
    public double[] NormalizedImportances()
    {
        double total = Importances.Sum();
        return total <= 0 ? Importances : Importances.Select(v => v / total).ToArray();
    }
}

// --- les decoupages ---------------------------------------------------------------
public static class Splits
{
    public sealed record Fold(int[] Train, int[] Test);

    /// <summary>
    /// Plis groupes et stratifies. Les sujets sont repartis en blocs : aucun sujet ne peut
    /// etre des deux cotes. La repartition est gloutonne — le plus gros sujet restant va au
    /// pli qui en contient le moins — ce qui approche le StratifiedGroupKFold de scikit-learn
    /// sans le reproduire a l'identique.
    /// </summary>
    public static List<Fold> GroupKFolds(int[] y, int[] groups, int k, int seed)
    {
        var rng = new Rng(seed);
        var ids = groups.Distinct().ToList();
        // Melange de Fisher-Yates, puis tri par effectif decroissant : les gros sujets d'abord.
        for (int i = ids.Count - 1; i > 0; i--)
        {
            int j = rng.NextInt(0, i + 1);
            (ids[i], ids[j]) = (ids[j], ids[i]);
        }
        var counts = ids.ToDictionary(g => g, g => Enumerable.Range(0, groups.Length).Count(i => groups[i] == g));
        ids = ids.OrderByDescending(g => counts[g]).ToList();

        var foldGroups = Enumerable.Range(0, k).Select(_ => new List<int>()).ToArray();
        var foldCounts = new int[k];
        foreach (int g in ids)
        {
            int target = Enumerable.Range(0, k).OrderBy(f => foldCounts[f]).First();
            foldGroups[target].Add(g);
            foldCounts[target] += counts[g];
        }

        var folds = new List<Fold>();
        for (int f = 0; f < k; f++)
        {
            var test = Enumerable.Range(0, groups.Length).Where(i => foldGroups[f].Contains(groups[i])).ToArray();
            var inTest = new HashSet<int>(foldGroups[f]);
            var train = Enumerable.Range(0, groups.Length).Where(i => !inTest.Contains(groups[i])).ToArray();
            folds.Add(new Fold(train, test));
        }
        return folds;
    }

    /// <summary>
    /// Plis stratifies naifs : les epochs sont repartis sans egard au sujet. C'est le
    /// decoupage fautif assume — il fait fuir les sujets entre apprentissage et test.
    /// </summary>
    public static List<Fold> StratifiedFolds(int[] y, int k, int seed)
    {
        var rng = new Rng(seed);
        var testSets = Enumerable.Range(0, k).Select(_ => new List<int>()).ToArray();
        foreach (int classe in y.Distinct())
        {
            var indices = Enumerable.Range(0, y.Length).Where(i => y[i] == classe).ToList();
            for (int i = indices.Count - 1; i > 0; i--)
            {
                int j = rng.NextInt(0, i + 1);
                (indices[i], indices[j]) = (indices[j], indices[i]);
            }
            for (int i = 0; i < indices.Count; i++) testSets[i % k].Add(indices[i]);
        }
        var folds = new List<Fold>();
        for (int f = 0; f < k; f++)
        {
            var test = testSets[f].ToArray();
            var inTest = new HashSet<int>(test);
            var train = Enumerable.Range(0, y.Length).Where(i => !inTest.Contains(i)).ToArray();
            folds.Add(new Fold(train, test));
        }
        return folds;
    }
}

// --- la procedure d'evaluation -----------------------------------------------------
public sealed record Hyperparameters(int MaxDepth, int MinLeaf);

public static class Decodeur
{
    public static readonly Hyperparameters[] GRID_COARSE =
    {
        new(int.MaxValue, 1),
        new(int.MaxValue, 4),
        new(8, 1),
        new(8, 4),
    };

    public static readonly Hyperparameters[] GRID_FINE =
        (from depth in new[] { int.MaxValue, 4, 8, 12 }
         from leaf in new[] { 1, 2, 4, 8 }
         select new Hyperparameters(depth, leaf)).ToArray();

    public static double Accuracy(RandomForest forest, double[][] X, int[] y, int[] indices)
    {
        if (indices.Length == 0) return double.NaN;
        int correct = 0;
        foreach (int i in indices)
        {
            if (forest.Predict(X[i]) == y[i]) correct++;
        }
        return (double)correct / indices.Length;
    }

    public static RandomForest Fit(double[][] X, int[] y, Hyperparameters hp, int seed)
        => new RandomForest(X, y, new ForestOptions
        {
            NTrees = N_TREES,
            MaxDepth = hp.MaxDepth,
            MinLeaf = hp.MinLeaf,
            Seed = seed,
        });

    public static int[] Slice(int[] source, int[] indices)
        => indices.Select(i => source[i]).ToArray();

    public static double[][] Slice(double[][] source, int[] indices)
        => indices.Select(i => source[i]).ToArray();

    /// <summary>Validation interne : moyenne des precisions des plis internes.</summary>
    public static double InnerScore(double[][] X, int[] y, int[] groups, Hyperparameters hp, int seed, bool aware)
    {
        var folds = aware ? Splits.GroupKFolds(y, groups, 3, seed) : Splits.StratifiedFolds(y, 3, seed);
        double total = 0.0;
        foreach (var fold in folds)
        {
            var forest = Fit(Slice(X, fold.Train), Slice(y, fold.Train), hp, seed);
            total += Accuracy(forest, X, y, fold.Test);
        }
        return total / folds.Count;
    }

    /// <summary>
    /// Precision en validation croisee imbriquee : quatre plis exterieurs, trois plis
    /// interieurs pour choisir les hyperparametres. La grille ne voit jamais le pli de test.
    /// </summary>
    public static double NestedCv(double[][] X, int[] y, int[] groups, int seed, bool aware = true,
        Hyperparameters[]? grid = null)
    {
        grid ??= GRID_COARSE;
        var outer = aware ? Splits.GroupKFolds(y, groups, 4, seed) : Splits.StratifiedFolds(y, 4, seed);
        double total = 0.0;
        foreach (var fold in outer)
        {
            var XTrain = Slice(X, fold.Train);
            var yTrain = Slice(y, fold.Train);
            var gTrain = Slice(groups, fold.Train);

            Hyperparameters best = grid[0];
            double bestScore = double.NegativeInfinity;
            foreach (var hp in grid)
            {
                double score = InnerScore(XTrain, yTrain, gTrain, hp, seed, aware);
                if (score > bestScore)
                {
                    bestScore = score;
                    best = hp;
                }
            }
            var finalForest = Fit(XTrain, yTrain, best, seed);
            total += Accuracy(finalForest, X, y, fold.Test);
        }
        return total / outer.Count;
    }

    /// <summary>
    /// Raccourci fautif, reproduit volontairement : la grille est choisie sur TOUTES les
    /// donnees, puis la performance est mesuree sur un decoupage groupe. Les hyperparametres
    /// ont vu le test — c'est le biais que la section 6 quantifie.
    /// </summary>
    public static (double accuracy, Hyperparameters best) NonNestedCv(double[][] X, int[] y, int[] groups,
        int seed, Hyperparameters[]? grid = null)
    {
        grid ??= GRID_COARSE;
        Hyperparameters best = grid[0];
        double bestScore = double.NegativeInfinity;
        foreach (var hp in grid)
        {
            double score = InnerScore(X, y, groups, hp, seed, aware: true);
            if (score > bestScore)
            {
                bestScore = score;
                best = hp;
            }
        }
        var outer = Splits.GroupKFolds(y, groups, 4, seed);
        double total = 0.0;
        foreach (var fold in outer)
        {
            var forest = Fit(Slice(X, fold.Train), Slice(y, fold.Train), best, seed);
            total += Accuracy(forest, X, y, fold.Test);
        }
        return (total / outer.Count, best);
    }
}

// Les trois vues des memes donnees : multi-region, HFD seul, LZC seul.
static double[][] Flatten(double[][][] feats)
{
    var rows = new double[feats.Length][];
    for (int i = 0; i < feats.Length; i++)
    {
        rows[i] = new double[N_ROI * 2];
        for (int r = 0; r < N_ROI; r++)
        {
            rows[i][2 * r] = feats[i][r][0];
            rows[i][2 * r + 1] = feats[i][r][1];
        }
    }
    return rows;
}

static double[][] Single(double[][][] feats, int idx)
{
    var rows = new double[feats.Length][];
    for (int i = 0; i < feats.Length; i++)
    {
        rows[i] = new double[N_ROI];
        for (int r = 0; r < N_ROI; r++) rows[i][r] = feats[i][r][idx];
    }
    return rows;
}

var X_MULTI = Flatten(cohort.Feats);
var X_HFD = Single(cohort.Feats, 0);
var X_LZC = Single(cohort.Feats, 1);

Console.WriteLine($"descripteurs : multi {X_MULTI[0].Length} | HFD {X_HFD[0].Length} | LZC {X_LZC[0].Length}");
Console.WriteLine($"foret : {N_TREES} arbres | grille grossiere {Decodeur.GRID_COARSE.Length} combinaisons | grille fine {Decodeur.GRID_FINE.Length}");


descripteurs : multi 24 | HFD 12 | LZC 12


foret : 100 arbres | grille grossiere 4 combinaisons | grille fine 16


### Lecture du resultat — la structure des plis

Cette table est le point de bascule du carnet, et elle se lit avant tout chiffre de precision.

Avec les **plis groupes**, la colonne « sujets communs » vaut **0** sur chaque pli : aucun sujet
n'est a la fois en apprentissage et en test. Le decodeur doit generaliser a un individu qu'il
n'a jamais vu.

Sans groupes, la meme colonne vaut **17** partout : tous les sujets sont des deux cotes. Deux
epochs du meme sujet partagent le meme decalage de persistance, le meme bruit propre, la meme
signature individuelle. Le decodeur apprend a **reconnaitre la personne**, ce qui est une tache
beaucoup plus facile que de reconnaitre la condition — et qui ne dit rien de ce qu'on veut
savoir.


In [6]:
// --- la structure des plis, mesuree avant toute precision --------------------------
static double[] ClassRate(int[] y, int[] idx) => new[] { idx.Length, idx.Count(i => y[i] == 1) == 0 ? 0.0 : (double)idx.Count(i => y[i] == 1) / idx.Length };

Console.WriteLine("plis groupes | pli | n test | positifs test | sujets test | sujets train | sujets communs");
foreach (var (aware, etiquette) in new[] { (true, "oui"), (false, "non") })
{
    var folds = aware
        ? Splits.GroupKFolds(cohort.Y, cohort.Groups, 4, 0)
        : Splits.StratifiedFolds(cohort.Y, 4, 0);
    for (int k = 0; k < folds.Count; k++)
    {
        var fold = folds[k];
        var sujetsTest = fold.Test.Select(i => cohort.Groups[i]).Distinct().ToHashSet();
        var sujetsTrain = fold.Train.Select(i => cohort.Groups[i]).Distinct().ToHashSet();
        int communs = sujetsTest.Intersect(sujetsTrain).Count();
        double positifs = (double)fold.Test.Count(i => cohort.Y[i] == 1) / fold.Test.Length;
        Console.WriteLine($"{etiquette,12} | {k,3} | {fold.Test.Length,6} | {positifs,13:F2} | {sujetsTest.Count,11} | {sujetsTrain.Count,12} | {communs,14}");
    }
}


plis groupes | pli | n test | positifs test | sujets test | sujets train | sujets communs


         oui |   0 |     60 |          0.53 |           5 |           12 |              0


         oui |   1 |     48 |          0.62 |           4 |           13 |              0


         oui |   2 |     48 |          0.50 |           4 |           13 |              0


         oui |   3 |     48 |          0.42 |           4 |           13 |              0


         non |   0 |     52 |          0.52 |          16 |           17 |             16


         non |   1 |     52 |          0.52 |          17 |           17 |             17


         non |   2 |     50 |          0.52 |          17 |           17 |             17


         non |   3 |     50 |          0.52 |          16 |           17 |             16


## 5. Q1 — le decodeur recupere-t-il la condition ?

Trois vues des memes donnees sont decodees separement : la dimension fractale seule, la
complexite de Lempel-Ziv seule, puis les deux combinees sur toutes les regions. Quatre graines
sont tirees pour chaque vue, parce qu'une precision unique ne dit rien sans sa dispersion.

La precision du hasard ici n'est pas 50 % par convention : les classes ne sont pas equilibrees
(98 placebo contre 106 LSD), donc un decodeur qui repond toujours la classe majoritaire
obtiendrait deja 52 %. C'est la **nulle empirique** de la section suivante qui donne le vrai
plancher a battre.


In [7]:
// --- Q1 : le decodeur retrouve-t-il la condition ? ----------------------------------
static (double mean, double sd, double min, double max) Summarize(IEnumerable<double> values)
{
    var arr = values.ToArray();
    double m = arr.Average();
    double sd = Math.Sqrt(arr.Sum(v => (v - m) * (v - m)) / arr.Length);
    return (m, sd, arr.Min(), arr.Max());
}

Console.WriteLine("modele | precision moyenne | ecart-type | min | max");
var accParFamille = new Dictionary<string, double[]>();
foreach (var (famille, X) in new[] { ("HFD", X_HFD), ("LZC", X_LZC), ("multi", X_MULTI) })
{
    var accs = SEEDS.Select(seed => Decodeur.NestedCv(X, cohort.Y, cohort.Groups, seed)).ToArray();
    accParFamille[famille] = accs;
    var (mean, sd, min, max) = Summarize(accs);
    Console.WriteLine($"{famille,6} | {mean,18:F3} | {sd,10:F3} | {min,5:F3} | {max,5:F3}");
}
Console.WriteLine();
Console.WriteLine($"taux de la classe majoritaire : {(double)cohort.Y.Count(v => v == 1) / cohort.Y.Length:F3}");


modele | precision moyenne | ecart-type | min | max


   HFD |              0.706 |      0.025 | 0.682 | 0.743


   LZC |              0.811 |      0.008 | 0.798 | 0.819


 multi |              0.715 |      0.021 | 0.681 | 0.738


taux de la classe majoritaire : 0.520


### Lecture du resultat — trois vues, et l'ordre ne surprend pas

Les trois vues decodent, mais pas au meme niveau : la complexite de Lempel-Ziv seule fait mieux
que la dimension fractale seule, et — c'est le point contre-intuitif — mieux que les deux
reunies. Les deux implementations donnent le meme ordre.

| vue | Python (moyenne +- ecart-type) | C# (moyenne +- ecart-type) |
|---|---|---|
| HFD seule | 0,694 +- 0,015 | 0,706 +- 0,025 |
| LZC seule | 0,770 +- 0,012 | 0,811 +- 0,008 |
| les deux | 0,743 +- 0,037 | 0,715 +- 0,021 |

La vue combinee perd ce que la meilleure famille gagnait. C'est la **dilution**, un effet connu
des forets : chaque coupure ne consulte qu'une racine carree des descripteurs, donc ajouter douze
descripteurs moins informatifs augmente la chance qu'une coupure tire un descripteur faible. Ce
n'est pas une propriete du cerveau etudie, c'est une propriete du decodeur — et elle se declare
plutot que de se taire.

L'ecart-type se lit aussi : 0,008 pour LZC, 0,025 pour HFD. La dimension fractale est un
estimateur plus instable — une pente ajustee sur dix-huit pas, qu'un seul epoch peut deplacer.
Cela fixe la resolution des comparaisons qui suivent : **un ecart inferieur a 0,03 ne se
distingue pas du bruit de graine**, et aucune conclusion de ce carnet ne s'appuie sur un ecart
de cet ordre.

### La nulle par permutation

La precision observee ne se compare pas a 50 %. Elle se compare a ce que **la meme procedure
complete** rend sur des etiquettes qui n'ont plus aucun lien avec les descripteurs.

La permutation melange les etiquettes, puis rejoue **toute** la chaine : decoupage groupe,
grille interne, foret, evaluation. Une nulle obtenue en reutilisant les hyperparametres deja
choisis sous-estimerait la variabilite de la procedure et fabriquerait un p trop petit.

Le p s'ecrit `(1 + #{nulle >= observee}) / (1 + n_perm)` : le « +1 » compte l'observation
elle-meme, et il fixe le plancher atteignable a `1 / (1 + n_perm)`. Avec trente permutations, on
ne peut donc **pas** descendre sous 0,032 — et un p egal a ce plancher veut dire « aucune
permutation n'a fait aussi bien », pas « p = 0 ».


In [8]:
// --- la nulle par permutation : la procedure complete est rejouee --------------------
static (double observed, double[] nulls, double p, double q95) PermutationNull(
    double[][] X, int[] y, int[] groups, int seed, int nPerm)
{
    var rng = new Rng(seed);
    double observed = Decodeur.NestedCv(X, y, groups, seed);
    var nulls = new double[nPerm];
    for (int b = 0; b < nPerm; b++)
    {
        // Fisher-Yates sur les etiquettes : les descripteurs et les groupes restent en place.
        var permuted = (int[])y.Clone();
        for (int i = permuted.Length - 1; i > 0; i--)
        {
            int j = rng.NextInt(0, i + 1);
            (permuted[i], permuted[j]) = (permuted[j], permuted[i]);
        }
        nulls[b] = Decodeur.NestedCv(X, permuted, groups, seed);
    }
    int atLeast = nulls.Count(v => v >= observed);
    double p = (1.0 + atLeast) / (1.0 + nPerm);
    var sorted = (double[])nulls.Clone();
    Array.Sort(sorted);
    double q95 = sorted[(int)Math.Ceiling(0.95 * nPerm) - 1];
    return (observed, nulls, p, q95);
}

var swPerm = System.Diagnostics.Stopwatch.StartNew();
var (OBSERVEE, NULLE, P_VALEUR, Q95) = PermutationNull(X_MULTI, cohort.Y, cohort.Groups, SEEDS[0], N_PERM);
swPerm.Stop();

Console.WriteLine($"precision observee (multi, graine 0) : {OBSERVEE:F3}");
Console.WriteLine($"nulle : moyenne {NULLE.Average():F3} | ecart-type {Math.Sqrt(NULLE.Sum(v => (v - NULLE.Average()) * (v - NULLE.Average())) / NULLE.Length):F3}");
Console.WriteLine($"quantile 95 de la nulle {Q95:F3} | max {NULLE.Max():F3}");
Console.WriteLine($"p = {P_VALEUR:F3} (plancher atteignable {1.0 / (1 + N_PERM):F3}) | {N_PERM} permutations en {swPerm.ElapsedMilliseconds} ms");
Console.WriteLine();
Console.WriteLine("nulle triee : " + string.Join(" ", NULLE.OrderBy(v => v).Select(v => v.ToString("F2"))));


precision observee (multi, graine 0) : 0.738


nulle : moyenne 0.477 | ecart-type 0.046


quantile 95 de la nulle 0.555 | max 0.584


p = 0.032 (plancher atteignable 0.032) | 30 permutations en 53780 ms


nulle triee : 0.39 0.40 0.42 0.42 0.43 0.43 0.43 0.44 0.45 0.45 0.46 0.46 0.46 0.47 0.48 0.48 0.48 0.48 0.48 0.49 0.49 0.50 0.51 0.52 0.53 0.53 0.54 0.54 0.56 0.58


## 6. Q2 — le biais du CV non imbrique

La comparaison est directe : la meme foret, les memes donnees, le meme decoupage groupe. Seule
change la **provenance des hyperparametres** — choisis dans chaque pli (imbriquee) ou choisis
une fois sur tout le jeu (raccourci).

Le raccourci est celui qu'on ecrit naturellement quand on veut aller vite : on regle le modele,
on regarde s'il marche, on publie le score. Ce que le score mesure alors, c'est la performance
d'un modele **dont les reglages ont ete choisis en connaissant les donnees de test**.

L'attendu theorique est simple : le raccourci surestime, et il surestime d'autant plus que la
grille est grande, puisque le maximum d'un plus grand nombre de combinaisons est plus chanceux.
C'est cet attendu que la cellule suivante met a l'epreuve.

### Lecture du resultat — le biais cherche n'apparait pas

| grille | Python : biais moyen [min ; max] | C# : biais moyen [min ; max] |
|---|---|---|
| grossiere (4 combinaisons) | +0,006 [-0,010 ; +0,036] | -0,002 [-0,015 ; +0,010] |
| fine (16 combinaisons) | -0,000 [-0,027 ; +0,031] | -0,008 [-0,021 ; +0,010] |

Le signe change d'une graine a l'autre, dans les deux implementations, et l'amplitude reste sous
l'ecart-type entre graines mesure en Q1. **Ce dispositif ne met donc pas le biais en evidence.**

Ce n'est pas un echec du carnet, c'est son resultat, et il vaut mieux qu'un chiffre arrange. Le
biais du raccourci existe en principe, mais il se paie en deux temps : il n'y a que quatre
combinaisons, et celle qui gagne sur l'ensemble est aussi celle qui convient a la plupart des
plis. Un maximum pris sur une petite grille n'achete pas une fraction mesurable de precision.

La consequence pratique est plus genante que rassurante. Un chercheur qui prendrait ce raccourci
sur ce jeu ne verrait **aucun signal d'alarme** dans ses chiffres : le biais serait present ou
absent sans que le score le trahisse. C'est la definition d'un defaut de procedure — il ne se
voit pas dans le resultat, il se voit dans le code.

Et la taille de grille ne tranche pas davantage : la grille fine ne montre pas un biais plus
grand que la grossiere, alors qu'elle quadruple le nombre de combinaisons essayees.

Ce resultat negatif doit donc se lire comme tel : **il ne prouve pas que le raccourci est
inoffensif**, il prouve qu'ici, a cette taille, avec cette grille, il ne se mesure pas. Le
danger qui se mesure vraiment, lui, est ailleurs — section suivante.


In [9]:
// --- Q2 : le biais du raccourci non imbrique ----------------------------------------
Console.WriteLine("grille | graine | imbriquee | non imbriquee | biais | hyperparametres choisis");
var biaisParGrille = new Dictionary<string, List<double>>();
foreach (var (nomGrille, grille) in new[] { ("grossiere (4)", Decodeur.GRID_COARSE), ("fine (16)", Decodeur.GRID_FINE) })
{
    biaisParGrille[nomGrille] = new List<double>();
    foreach (int seed in SEEDS)
    {
        double imbriquee = Decodeur.NestedCv(X_MULTI, cohort.Y, cohort.Groups, seed, true, grille);
        var (raccourci, meilleurs) = Decodeur.NonNestedCv(X_MULTI, cohort.Y, cohort.Groups, seed, grille);
        double biais = raccourci - imbriquee;
        biaisParGrille[nomGrille].Add(biais);
        string depth = meilleurs.MaxDepth == int.MaxValue ? "None" : meilleurs.MaxDepth.ToString();
        Console.WriteLine($"{nomGrille,14} | {seed,6} | {imbriquee,9:F3} | {raccourci,13:F3} | {biais,5:F3} | profondeur {depth}, feuille {meilleurs.MinLeaf}");
    }
}
Console.WriteLine();
foreach (var (grille, values) in biaisParGrille)
{
    Console.WriteLine($"biais {grille} : moyenne {values.Average():F3} | min {values.Min():F3} | max {values.Max():F3}");
}


grille | graine | imbriquee | non imbriquee | biais | hyperparametres choisis


 grossiere (4) |      0 |     0.738 |         0.748 | 0.010 | profondeur 8, feuille 4


 grossiere (4) |      1 |     0.681 |         0.677 | -0.004 | profondeur None, feuille 4


 grossiere (4) |      7 |     0.721 |         0.720 | -0.001 | profondeur None, feuille 1


 grossiere (4) |     42 |     0.720 |         0.705 | -0.015 | profondeur 8, feuille 4


     fine (16) |      0 |     0.753 |         0.748 | -0.005 | profondeur 8, feuille 4


     fine (16) |      1 |     0.698 |         0.677 | -0.021 | profondeur None, feuille 4


     fine (16) |      7 |     0.705 |         0.688 | -0.018 | profondeur 8, feuille 2


     fine (16) |     42 |     0.705 |         0.716 | 0.010 | profondeur 8, feuille 8


biais grossiere (4) : moyenne -0.002 | min -0.015 | max 0.010


biais fine (16) : moyenne -0.008 | min -0.021 | max 0.010


## 7. Q3 — la fuite de groupe

Meme grille, meme foret, meme jeu : seule change la **repartition des sujets**. Avec groupes,
un sujet est entierement d'un cote. Sans groupes, ses epochs se repartissent.

C'est la difference la plus facile a rater en pratique, parce que le code est presque le meme :
`StratifiedKFold` au lieu de `StratifiedGroupKFold`, un argument oublie. Le chiffre obtenu reste
plausible — c'est ce qui rend la fuite dangereuse. Il ne mesure simplement plus la meme chose.

Le signe attendu : la version non groupee **surestime**. Elle repond a « ce sujet est-il
reconnaissable ? », question a laquelle la reponse est oui par construction.

### Lecture du resultat — ici, l'ecart se mesure

| implementation | ecart moyen | min | max |
|---|---|---|---|
| Python | +0,082 | +0,047 | +0,115 |
| C# | +0,143 | +0,101 | +0,167 |

Le signe est le meme sur **les huit graines** des deux implementations : la version non groupee
surestime partout, de quatre a dix-sept points selon la graine. Compare a la resolution de 0,03
etablie en Q1, cet ecart n'est pas du bruit : c'est un effet de protocole, plus grand que l'effet
neurobiologique que le carnet cherche a detecter.

C'est le contraste qui compte. Le biais de la grille non imbriquee (Q2) ne s'est pas mesure ; la
fuite de groupe, elle, se mesure sur chaque graine. Les deux defauts sont voisins dans le code et
de poids tres differents dans les chiffres — et c'est le plus facile a commettre par inadvertance
(un argument par defaut) qui coute le plus cher.

La cellule de structure des plis disait deja pourquoi : avec les plis non groupes, chaque pli de
test partage seize ou dix-sept sujets sur dix-sept avec son pli d'apprentissage. Le decodeur n'a
pas de nouvelle personne a qui generaliser — il n'y a plus de generalisation a mesurer.


In [10]:
// --- Q3 : la fuite de groupe --------------------------------------------------------
Console.WriteLine("graine | plis groupes | plis non groupes | ecart");
var ecarts = new List<double>();
foreach (int seed in SEEDS)
{
    double groupee = Decodeur.NestedCv(X_MULTI, cohort.Y, cohort.Groups, seed, aware: true);
    double fuite = Decodeur.NestedCv(X_MULTI, cohort.Y, cohort.Groups, seed, aware: false);
    ecarts.Add(fuite - groupee);
    Console.WriteLine($"{seed,6} | {groupee,12:F3} | {fuite,16:F3} | {fuite - groupee,5:F3}");
}
Console.WriteLine();
Console.WriteLine($"ecart moyen {ecarts.Average():F3} | min {ecarts.Min():F3} | max {ecarts.Max():F3}");


graine | plis groupes | plis non groupes | ecart


     0 |        0.738 |            0.838 | 0.101


     1 |        0.681 |            0.842 | 0.161


     7 |        0.721 |            0.888 | 0.167


    42 |        0.720 |            0.863 | 0.143


ecart moyen 0.143 | min 0.101 | max 0.167


## 8. Q4 — le seuil de puissance

Une validation honnete ne suffit pas a rendre un resultat interessant : encore faut-il que
l'effet existe. Le terrain permet de le balayer — on fait varier l'amplitude `eta` injectee au
pole posterieur, de zero (aucun effet) a la valeur du carnet.

Ce balayage est le **controle negatif** du carnet. A `eta = 0`, les deux conditions sont des
tirages du meme processus : le decodeur doit retomber dans la nulle. Si une precision elevee
apparaissait a `eta = 0`, c'est la procedure qui fabriquerait l'effet, pas les donnees.

Le point de bascule — l'amplitude a partir de laquelle la precision depasse durablement le
quantile 95 de la nulle — est la **sensibilite** de la procedure : ce qu'elle peut detecter sur
vingt-quatre descripteurs et 204 epochs.


In [11]:
// --- Q4 : le seuil de puissance, et le controle negatif a eta = 0 -------------------
double[] amplitudes = { 0.0, 0.02, 0.04, 0.06 };
Console.WriteLine("eta | precision moyenne | graine 0 | graine 1");
var moyenneParEta = new List<double>();
foreach (double eta in amplitudes)
{
    var accs = new List<double>();
    // Deux graines suffisent ici : on cherche une tendance, pas un verdict multi-graine.
    foreach (int seed in SEEDS.Take(2))
    {
        var c = Terrain.Simulate(eta, seed);
        accs.Add(Decodeur.NestedCv(Flatten(c.Feats), c.Y, c.Groups, seed));
    }
    moyenneParEta.Add(accs.Average());
    Console.WriteLine($"{eta,4:F2} | {accs.Average(),18:F3} | {accs[0],8:F3} | {accs[1],8:F3}");
}
Console.WriteLine();
Console.WriteLine($"quantile 95 de la nulle (Q1) : {Q95:F3} | hasard : 0.500");
Console.WriteLine("depassements du quantile 95 : " + string.Join(", ",
    amplitudes.Select((eta, i) => $"{eta:F2} -> {(moyenneParEta[i] > Q95 ? "oui" : "non")}")));


eta | precision moyenne | graine 0 | graine 1


0.00 |              0.468 |    0.453 |    0.483


0.02 |              0.570 |    0.532 |    0.607


0.04 |              0.641 |    0.625 |    0.656


0.06 |              0.739 |    0.738 |    0.740


quantile 95 de la nulle (Q1) : 0.555 | hasard : 0.500


depassements du quantile 95 : 0.00 -> non, 0.02 -> oui, 0.04 -> oui, 0.06 -> oui


## 9. Ce que le decodeur regarde

Une precision dit **si** le decodeur separe, pas **sur quoi**. L'importance d'impurete — la
baisse de desequilibre apportee par chaque descripteur, cumulee sur les cent arbres — repond a
la seconde question.

Le motif attendu suit le terrain : l'importance devrait se concentrer sur les regions
**posterieures**, celles que le LSD desynchronise, et rester faible sur la region 0, le temoin
sans effet. La cellule suivante met cet attendu a l'epreuve.

### Lecture du resultat — la region posterieure en tete, mais pas seule

La region 11 arrive bien en tete : 0,163 ici, 0,187 dans la version Python. Le decodeur regarde
la ou l'effet a ete injecte.

Mais la region 0 — celle ou **aucun effet n'a ete injecte** — n'est pas faible du tout : 0,102
ici, 0,09 en Python, contre 0,042 si les vingt-quatre descripteurs se partageaient l'importance a
parts egales. Les deux implementations reproduisent ce motif : ce n'est pas un accident de tirage.

C'est une propriete de l'importance d'impurete, et elle merite d'etre dite. Une foret partage la
baisse d'impurete entre tous les descripteurs qu'elle examine : un descripteur consulte tot, sur
un noeud encore impur, encaisse une part de cette baisse **meme s'il ne porte aucun signal**. Le
classement brut ne distingue donc pas « ce descripteur separe les classes » de « ce descripteur a
ete examine sur un noeud impur ».

La consequence est directe : une figure d'importance sans nulle ne prouve pas ce qu'elle a l'air
de prouver. Que la region 11 sorte en tete est necessaire, ce n'est pas suffisant — la region 0
sort au-dessus du hasard sans porter d'effet.

D'ou la correction que le papier applique et que l'exercice 3 fait ecrire : permuter les
etiquettes, reentrainer, lire l'importance que la procedure attribue au hasard, et la retrancher.
Ce qui reste apres soustraction est l'importance imputable au signal.

Une reserve de plus, qui vaut pour toute la litterature sur ce point : l'importance d'impurete
est biaisee en faveur des descripteurs a nombreuses valeurs distinctes. Elle se lit comme un
profil, jamais comme un classement absolu.


In [12]:
// --- ce que le decodeur regarde : importance d'impurete par region ------------------
var foretComplete = new RandomForest(X_MULTI, cohort.Y, new ForestOptions { NTrees = 300, Seed = 0 });
var importances = foretComplete.NormalizedImportances();

Console.WriteLine("region | importance HFD | importance LZC | total");
for (int r = 0; r < N_ROI; r++)
{
    double hfd = importances[2 * r];
    double lzc = importances[2 * r + 1];
    Console.WriteLine($"{r,6} | {hfd,14:F4} | {lzc,14:F4} | {hfd + lzc,5:F4}");
}
var sommeHfd = Enumerable.Range(0, N_ROI).Sum(r => importances[2 * r]);
var sommeLzc = Enumerable.Range(0, N_ROI).Sum(r => importances[2 * r + 1]);
Console.WriteLine();
Console.WriteLine($"importance totale HFD {sommeHfd:F3} | LZC {sommeLzc:F3}");
Console.WriteLine($"region 0 (temoin sans effet) {importances[0] + importances[1]:F3} | region posterieure 11 {importances[2 * (N_ROI - 1)] + importances[2 * (N_ROI - 1) + 1]:F3}");


region | importance HFD | importance LZC | total


     0 |         0.0425 |         0.0597 | 0.1022


     1 |         0.0366 |         0.0392 | 0.0758


     2 |         0.0278 |         0.0332 | 0.0610


     3 |         0.0263 |         0.0235 | 0.0498


     4 |         0.0254 |         0.0346 | 0.0599


... (5 outputs left out)

    10 |         0.0327 |         0.0607 | 0.0934


    11 |         0.0454 |         0.1178 | 0.1631


importance totale HFD 0.390 | LZC 0.610


region 0 (temoin sans effet) 0.102 | region posterieure 11 0.163


## Exercices

Les trois exercices reprennent les trois questions du carnet, mais il faut ecrire le corps
soi-meme. Chaque cellule definit une fonction et l'appelle : le carnet s'execute de bout en bout,
meme incomplet.

### Exercice 1 — decoder a une autre amplitude

Ecrire `DecoderAtEta` : simuler une cohorte a l'amplitude demandee, puis rendre la precision en
validation croisee imbriquee. Le decodage doit rester groupe et stratifie — c'est le point de
l'exercice, pas un detail.

Attendu : la precision decroit vers la nulle quand l'amplitude tend vers zero.


In [13]:
// TODO etudiant : completer le corps de cette fonction.
// Indice : Terrain.Simulate(eta, seed) rend une cohorte ; Flatten(...) en fait la matrice
// multi-region ; Decodeur.NestedCv(...) rend la precision imbriquee.
static double DecoderAtEta(double eta, int seed)
{
    // Etape 1 : simuler la cohorte a cette amplitude.
    // Etape 2 : aplatir les descripteurs.
    // Etape 3 : rendre la precision en validation croisee imbriquee.
    return double.NaN;
}

Console.WriteLine($"precision a eta = 0.03 : {DecoderAtEta(0.03, 0):F3} (a completer)");


precision a eta = 0.03 : NaN (a completer)


### Exercice 2 — la fuite selon l'amplitude

Ecrire `LeakCurve` : pour plusieurs amplitudes, rendre l'ecart entre la precision non groupee
et la precision groupee.

Attendu : a `eta = 0` (aucun effet), l'ecart doit rester **positif et net**. C'est la signature
de la fuite : meme sans effet a detecter, le decoupage non groupe trouve quelque chose, parce
qu'il reconnait les sujets.


In [14]:
// TODO etudiant : completer le corps de cette fonction.
// Indice : deux appels a Decodeur.NestedCv sur la meme cohorte, avec aware: false puis true.
static List<(double eta, double ecart)> LeakCurve(double[] amplitudes, int seed)
{
    var lignes = new List<(double eta, double ecart)>();
    foreach (double eta in amplitudes)
    {
        // Etape 1 : simuler la cohorte a cette amplitude.
        // Etape 2 : mesurer les deux precisions — sans groupes, puis avec groupes.
        // Etape 3 : ajouter l'ecart (non groupee moins groupee) a la liste.
    }
    return lignes;
}

foreach (var (eta, ecart) in LeakCurve(new[] { 0.0, 0.03, 0.06 }, 0))
{
    Console.WriteLine($"eta {eta:F2} | ecart {ecart:F3}");
}
Console.WriteLine("(a completer : la liste doit rester vide tant que le corps n'est pas ecrit)");


(a completer : la liste doit rester vide tant que le corps n'est pas ecrit)


### Exercice 3 — la nulle d'importance du papier

Ecrire `ImportanceByLabelPermutation` : permuter les etiquettes, reentrainer, et mesurer
l'importance. Repeter, puis rendre l'importance moyenne **moins** celle obtenue sur les vraies
etiquettes : ce qui reste est l'importance que la procedure attribue au hasard.

C'est la correction que le papier applique a sa figure d'importance : sans elle, meme un
descripteur sans effet recoit une part non nulle, parce qu'une foret partage toujours l'impurete
entre les variables qu'elle examine.


In [15]:
// TODO etudiant : completer le corps de cette fonction.
// Indice : new RandomForest(X, yPermute, options).NormalizedImportances() rend le profil
// d'une permutation ; la moyenne sur n_perm se retranche au profil des vraies etiquettes.
static double[] ImportanceByLabelPermutation(double[][] X, int[] y, int nPerm, int seed)
{
    var rng = new Rng(seed);
    var moyenne = new double[X[0].Length];
    for (int b = 0; b < nPerm; b++)
    {
        // Etape 1 : permuter les etiquettes.
        // Etape 2 : entrainer une foret et lire les importances normalisees.
        // Etape 3 : accumuler.
    }
    // Etape 4 : rendre la moyenne (a completer).
    return moyenne;
}

var corrigees = ImportanceByLabelPermutation(X_MULTI, cohort.Y, 5, 0);
Console.WriteLine($"importance corrigee de la region 11 : {corrigees[2 * (N_ROI - 1)] + corrigees[2 * (N_ROI - 1) + 1]:F4} (a completer)");


importance corrigee de la region 11 : 0.0000 (a completer)


## 10. Limites et suite

**Le portage n'est pas un jumeau numerique.** Les valeurs de ce carnet sont celles de sa propre
foret et de son propre generateur ; elles ne se comparent pas chiffre a chiffre a celles du
carnet Python. Ce qui se compare est la structure : la fuite surestime, la nulle encadre
l'observation, le controle negatif retombe au hasard.

**La foret est volontairement simple.** Seize quantiles par descripteur et par noeud, pas toutes
les coupures possibles ; pas d'echantillonnage sans remise des descripteurs au-dela de la racine
carree. C'est suffisant pour que les questions methodologiques s'entendent, et insuffisant pour
pretendre a une precision de pointe.

**Le decoupage groupe est glouton**, pas celui de scikit-learn : la repartition des sujets est
proche, pas identique. Les chiffres de Q1 et Q3 en dependent a la marge — la comparaison entre
variantes, elle, reste valable parce que les deux variantes partagent le meme decoupage.

**Ce que le carnet ne fait pas** : aucun test de significativite sur les ecarts de Q2 et Q3. Le
biais est montre sur quatre graines, avec sa dispersion ; il n'est pas teste. Un carnet qui
affirmerait « le biais est significatif » a partir de quatre valeurs le ferait sans instrument.

**Le resultat negatif de Q2 ne se generalise pas.** Il dit qu'ici, a vingt-quatre descripteurs et
204 epochs, avec quatre puis seize combinaisons, le raccourci ne se mesure pas. Il ne dit rien
d'un jeu plus grand, d'une grille plus large, ou d'un modele plus souple que la foret — c'est
precisement dans ces cas que le biais du raccourci a ete documente.


## Verdict

Deux implementations, meme protocole, memes conclusions de structure :

| Question | Mesure | Verdict |
|---|---|---|
| **Q1** — le decodeur retrouve-t-il la condition ? | LZC 0,81 (C#) / 0,77 (Python) contre une nulle a 95 % de 0,555 / 0,552 ; p au plancher sur trente permutations | **oui**, et nettement au-dessus de ce que la procedure rend sur des etiquettes melangees |
| **Q2** — le raccourci non imbrique surestime-t-il ? | biais -0,002 a -0,008 (C#), +0,006 a -0,000 (Python), signe instable entre graines | **non mesurable ici** — l'attendu theorique ne se retrouve pas a cette taille de grille |
| **Q3** — la fuite de groupe fabrique-t-elle l'effet ? | +0,143 (C#) / +0,082 (Python), positif sur les huit graines | **oui**, et c'est le defaut le plus couteux du carnet |
| **Q4** — le controle negatif tient-il ? | 0,468 a `eta = 0` (C#) / 0,498 (Python), sous le quantile 95 ; 0,739 / 0,772 a l'amplitude nominale | **oui** : sans effet injecte, la procedure retombe au hasard |
| **Section 9** — sur quoi le decodeur s'appuie-t-il ? | region 11 en tete, mais region 0 a 0,102 contre 0,042 attendus | **partiellement** : le classement brut ne suffit pas, la nulle d'importance est requise |

La lecon du carnet se lit en deux temps, et le second est le plus utile.

**Une precision elevee ne dit rien tant qu'on n'a pas montre ce que la meme procedure rend sur
des donnees sans effet.** C'est ce que la nulle de Q4 etablit, et c'est ce qui rend les chiffres
de Q1 interpretables.

**Et un defaut de procedure ne se voit pas toujours dans le resultat.** Le biais du CV non
imbrique, celui dont parle toute la litterature, ne s'est pas mesure ici — dans aucune des deux
implementations. La fuite de groupe, elle, a coute quatre a dix-sept points sur chaque graine.
Les deux tiennent en un argument qui change, et c'est le plus discrete qui etait le plus
dangereux.

Ce que le carnet ne dit pas : que le raccourci non imbrique est inoffensif. Il dit qu'a cette
taille, sur ce jeu, avec quatre puis seize combinaisons, il ne se mesure pas — et qu'un score
seul n'aurait donc pas suffi a le trahir.

Le motif d'importance non plus ne se lit pas seul : la region 0 sort au-dessus du hasard sans
porter aucun effet. Sans la permutation de l'exercice 3, la figure d'importance serait une
promesse que les donnees ne tiennent pas.


## Pour aller plus loin

- `ICT-36-FLens-FactoredGeometry-Python.ipynb` — la theorie de la lentille de facteurs dont ce
  carnet tire ses descripteurs.
- `ICT-36c-FLens-HiguchiBridge-Python.ipynb` — le pont entre la dimension de Higuchi et la
  geometrie du signal : pourquoi la perte de persistance augmente la dimension.
- `ICT-37-FLens-BeliefState-Python.ipynb` — ce que le decodeur fait quand la variable latente
  n'est pas directement observable.
- La serie `Probas/` pour les outils statistiques qui manquent ici : intervalles de confiance
  sur une precision, tests apparies entre deux procedures.
